# EV fleet with explicit process-product paradigm

This notebook models the EV fleet directly with **explicit process/product nodes** and TDs on production and inputs.

In [ ]:
import bw2data as bd
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution as TD
import numpy as np

## Build explicit process/product fleet model
No intermediate `fleet_service` node; FU is the fleet product directly.

In [ ]:
fg = bd.Database("foreground")

fleet_process = fg.new_node(code="fleet_driving_process_explicit", name="fleet driving process (explicit)", unit="vkm")
fleet_process.save()

fleet_product = fg.new_node(code="fleet_driving_product_explicit", name="fleet driving product (explicit)", unit="vkm", type="product")
fleet_product.save()

cohort_years = np.arange(2025, 2041)
cohort_amounts = np.ones(len(cohort_years))/len(cohort_years)
cohort_td = TD(date=np.array([np.datetime64(f"{y}-01-01") for y in cohort_years]), amount=cohort_amounts)

fleet_process.new_edge(input=fleet_product, type="production", amount=1.0, temporal_distribution=cohort_td).save()

In [ ]:
electricity = bd.get_node(database="background_2020", code="electricity")
ev_prod = bd.get_node(database="foreground", code="ev_production")
used_ev = bd.get_node(database="foreground", code="used_ev")

age_td = TD(date=np.array([np.timedelta64(y, "Y") for y in [0,1,2,3]]), amount=np.array([0.30,0.27,0.23,0.20]))
retirement_td = TD(date=np.array([np.timedelta64(y, "Y") for y in [12,13,14,15]]), amount=np.array([0.20,0.30,0.30,0.20]))

el = fleet_process.new_edge(input=electricity, type="technosphere", amount=1.0, temporal_distribution=age_td)
el["temporal_evolution_factors"] = {2025: 1.0, 2030: 0.9, 2035: 0.82, 2040: 0.75}
el["temporal_evolution_reference"] = "consumer"
el.save()

fleet_process.new_edge(input=ev_prod, type="technosphere", amount=1.0, temporal_distribution=age_td).save()
fleet_process.new_edge(input=used_ev, type="technosphere", amount=1.0, temporal_distribution=retirement_td).save()

In [ ]:
method = ("IPCC 2021", "climate change", "global warming potential (GWP100)")
database_dates = {"background_2020": "2020", "background_2040": "2040", "foreground": "dynamic"}

tlca = TimexLCA({fleet_product: 1}, method, database_dates)
tlca.build_timeline(starting_datetime="2024-01-01", temporal_grouping="year")
tlca.lci()
tlca.dynamic_lcia()
print(tlca.dynamic_score)
tlca.timeline.head()